In [2]:
import yaml
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer, OneHotEncoder, OrdinalEncoder

# PRUEBAS build_preprocessor

In [3]:
config_file = '../config/01_experimento.yaml'  

with open(config_file, 'r') as f:
    config = yaml.safe_load(f)

columnas_config = config.get('columnas', {})
preprocessor_config = config.get('preprocessing', {})

print("=== Configuración cargada ===")
print("columnas_config keys:", columnas_config.keys())
print("preprocessor_config:", preprocessor_config)

=== Configuración cargada ===
columnas_config keys: dict_keys(['num_cols', 'cat_ord_cols', 'cat_nom_ohe_drop', 'cat_nom_ohe', 'cat_nom_frec'])
preprocessor_config: {'remainder': 'passthrough', 'onehot_drop': {'drop': 'first', 'handle_unknown': 'ignore'}, 'onehot': {'handle_unknown': 'ignore'}}


## Datos

In [4]:
data_prueba = pd.DataFrame({
    'age': [25, 45, 60],
    'annual_income': [30000, 50000, 40000],
    'monthly_income': [2500, 4167, 3333],
    'debt_to_income_ratio': [0.2, 0.3, 0.25],
    'credit_score': [700, 650, 720],
    'loan_amount': [10000, 20000, 15000],
    'interest_rate': [5.5, 6.0, 5.0],
    'installment': [300, 500, 400],
    'num_of_open_accounts': [3, 5, 4],
    'total_credit_limit': [5000, 8000, 6000],
    'current_balance': [2000, 3000, 2500],
    'delinquency_history': [0, 2, 1],
    'public_records': [0, 1, 0],
    'num_of_delinquencies': [0, 2, 1],
    'education_level': ['Bachelor', 'Master', 'High School'],
    'grade_subgrade': ['A2', 'B1', 'A1'],
    'gender': ['Male', 'Female', 'Other'],
    'marital_status': ['Single', 'Married', 'Divorced'],
    'employment_status': ['Employed', 'Unemployed', 'Self-employed'],
    'loan_purpose': ['Education', 'Home', 'Car'],
    'loan_term': [36, 60, 36]  # si es numérica, si es texto cámbialo
})

print("\n=== Datos de prueba ===")
print(data_prueba.head())


=== Datos de prueba ===
   age  annual_income  monthly_income  debt_to_income_ratio  credit_score  \
0   25          30000            2500                  0.20           700   
1   45          50000            4167                  0.30           650   
2   60          40000            3333                  0.25           720   

   loan_amount  interest_rate  installment  num_of_open_accounts  \
0        10000            5.5          300                     3   
1        20000            6.0          500                     5   
2        15000            5.0          400                     4   

   total_credit_limit  ...  delinquency_history  public_records  \
0                5000  ...                    0               0   
1                8000  ...                    2               1   
2                6000  ...                    1               0   

   num_of_delinquencies  education_level grade_subgrade  gender  \
0                     0         Bachelor             A2  

## build_preprocessor

In [ ]:
def build_preprocessor(columnas_config, preprocessor_config):

    # Columnas nuevas de ingenieria de características
    num_cols = columnas_config.get('num_cols', {})
    cat_ord_cols = columnas_config.get('cat_ord_cols', {})
    cat_nom_ohe_drop = columnas_config.get('cat_nom_ohe_drop', [])
    cat_nom_ohe = columnas_config.get('cat_nom_ohe', [])
    #cat_nom_frec = columnas_config.get('cat_nom_frec', [])

    transformers = []

    # Pipeline para datos numéricos
    transform_groups = {}
    for col, conf in num_cols.items():
        t = conf.get('transform', 'passthrough')
        transform_groups.setdefault(t, []).append(col)
    
    for t, cols in transform_groups.items():
        if t == 'passthrough':
            transformers.append((f'num_passthrough_{len(transformers)}', 'passthrough', cols))

        elif t == 'scale':
            pipeline = Pipeline([('scaler', StandardScaler())])
            transformers.append((f'scale_{len(transformers)}', pipeline, cols))

        elif t == 'log_scale':
            pipeline = Pipeline([
                ('log', FunctionTransformer(np.log, validate=True)),
                ('scaler', StandardScaler())
                ])
            transformers.append((f'log_scale_{len(transformers)}', pipeline, cols))

        elif t == 'log1p_scale':
            pipeline = Pipeline([
                ('log1p', FunctionTransformer(np.log1p, validate= True)),
                ('scaler', StandardScaler())])
            transformers.append((f'log1p_scale_{len(transformers)}', pipeline, cols))

        else:
            transformers.append((f'no_reconocida_{t}_{len(transformers)}', 'passthrough', cols)) 

    # Pipeline para datos categóricos ordinales
    if cat_ord_cols:
        ord_cols = list(cat_ord_cols.keys())
        ord_categories = [cat_ord_cols[col]['categories'] for col in ord_cols]
        ord_pipeline = Pipeline([('ord_encoder', OrdinalEncoder(categories=ord_categories))])
        transformers.append((f'ord_encoder_{len(transformers)}', ord_pipeline, ord_cols))

    # Pipeline para datos categóricos nominales ohe_dop
    if cat_nom_ohe_drop:
        ohe_drop_config = preprocessor_config.get('onehot_drop', {})
        cat_ohe_drop_pipeline = Pipeline([('ohe_drop', OneHotEncoder(
            drop = ohe_drop_config.get('drop', 'first'),
            handle_unknown = ohe_drop_config.get('handle_unknown', 'ignore'),
            sparse_output = False))
            ])
        transformers.append((f'ohe_drop_{len(transformers)}', cat_ohe_drop_pipeline, cat_nom_ohe_drop))
    
    # Pipeline para datos categoricos nominales ohe
    if cat_nom_ohe:
        ohe_config = preprocessor_config.get('onehot', {})
        ohe_pipeline = Pipeline([('ohe', OneHotEncoder(
            handle_unknown = ohe_config.get('handle_unknown', 'ignore'),
            sparse_output = False))
            ])
        transformers.append((f'ohe_{len(transformers)}', ohe_pipeline, cat_nom_ohe))


    processor = ColumnTransformer(
        transformers = transformers,
        remainder = preprocessor_config.get('remainder', 'drop')
    )
    return processor


build_preprocessor(columnas_config, preprocessor_config)


ColumnTransformer(remainder='passthrough',
                  transformers=[('num_passthrough_0', 'passthrough',
                                 ['age', 'payment_income']),
                                ('log_scale_1',
                                 Pipeline(steps=[('log',
                                                  FunctionTransformer(func=<ufunc 'log'>,
                                                                      validate=True)),
                                                 ('scaler', StandardScaler())]),
                                 ['annual_income', 'monthly_income',
                                  'debt_to_income_ratio', 'total_credit_limit',
                                  'current_balance']),
                                ('scale_2',
                                 Pi...
                                                                              'C1',
                                                                              'C2',
                                                                              'D1',
                                                                              'D2',
                                                                              'E1',
                                                                              'E2',
                                                                              'F1',
                                                                              'F2',
                                                                              'G1'],
                                                                             ['joven',
                                                                              'adulto_joven',
                                                                              'adulto',
                                                                              'adulto_mayor',
                                                                              '3_Edad']]))]),
                                 ['education_level', 'grade_subgrade',
                                  'age_group']),
                                ('ohe_drop',
                                 Pipeline(steps=[('ohe_drop',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['gender', 'marital_status',
                                  'employment_status', 'loan_purpose',
                                  'loan_term'])])